# 5. 多通道 Retrieval Budget Fusion：怎样按意图分配候选预算并做可比的排名融合？

## 面试回答主线

关键词、dense 和知识图谱检索擅长的查询不同，固定平均切分预算会浪费通道能力。原始 BM25、cosine 和图距离也不在同一量纲，不能直接比较或相加；应先按每通道内部 rank 归一，再用 RRF 或校准模型融合。面试时我会为真实精确编号、概念问答和关系查询分配不同候选数，用同一批通道结果比较 raw-score 基线与 weighted RRF。融合必须按 doc_id 去重，并在某通道为空时把未使用预算回填给可用通道。结果表应展示每条 query 的预算、候选账本和最终文档，而不是只给平均 Recall。生产系统还要把延迟、取消、过滤条件和上下文 token 预算一起考虑。

## 1. 真实案例：六种查询、九篇文档与三通道候选

精确查询依赖订单号或错误码，概念查询依赖语义近邻，关系查询依赖图路径。通道输出保留真实标题与各自原始分数尺度：lexical 可到十几分，dense 在 0～1，graph 用路径相关度。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示通道候选与融合账本
documents = {"D1": "订单 EX2048 的退款进度", "D2": "用来源引用降低 RAG 幻觉", "D3": "退款审批人与财务关系", "D4": "E_CONN_RESET 连接重置排障", "D5": "向量索引分片与量化优化", "D6": "密钥泄露后的通知责任人", "D7": "通用退款帮助", "D8": "数据库常见问题", "D9": "大模型回答质量概览"}  # 定义九篇具有真实业务语义的候选文档
queries = [{"id": "F01", "text": "订单 EX2048 退款到哪了", "intent": "exact", "expected": "D1", "channels": {"lexical": [("D1", 12.0), ("D7", 7.0), ("D9", 3.0)], "dense": [("D7", 0.91), ("D1", 0.88), ("D9", 0.61)], "graph": [("D1", 0.50), ("D3", 0.33), ("D7", 0.25)]}}, {"id": "F02", "text": "怎样减少 RAG 编造", "intent": "concept", "expected": "D2", "channels": {"lexical": [("D9", 10.0), ("D2", 6.0), ("D7", 3.0)], "dense": [("D2", 0.95), ("D9", 0.72), ("D5", 0.60)], "graph": [("D9", 0.50), ("D2", 0.45), ("D5", 0.20)]}}, {"id": "F03", "text": "退款最终需要谁审批", "intent": "relation", "expected": "D3", "channels": {"lexical": [("D7", 11.0), ("D1", 7.0), ("D3", 5.0)], "dense": [("D7", 0.88), ("D3", 0.86), ("D1", 0.70)], "graph": [("D3", 1.00), ("D1", 0.40), ("D7", 0.30)]}}, {"id": "F04", "text": "错误码 E_CONN_RESET", "intent": "exact", "expected": "D4", "channels": {"lexical": [("D4", 15.0), ("D8", 6.0), ("D9", 2.0)], "dense": [("D4", 0.94), ("D8", 0.80), ("D9", 0.50)], "graph": [("D8", 0.50), ("D4", 0.45), ("D9", 0.20)]}}, {"id": "F05", "text": "向量检索太慢怎样扩展", "intent": "concept", "expected": "D5", "channels": {"lexical": [("D8", 9.0), ("D5", 6.0), ("D9", 3.0)], "dense": [("D5", 0.96), ("D8", 0.70), ("D9", 0.55)], "graph": [("D8", 0.50), ("D5", 0.46), ("D9", 0.20)]}}, {"id": "F06", "text": "密钥泄露后应该通知谁", "intent": "relation", "expected": "D6", "channels": {"lexical": [("D9", 10.0), ("D6", 6.0), ("D8", 3.0)], "dense": [("D9", 0.87), ("D6", 0.85), ("D8", 0.60)], "graph": [("D6", 1.00), ("D9", 0.40), ("D8", 0.30)]}}]  # 定义六个查询的真实多通道候选与不同量纲分数
queries[1]["channels"]["graph"] = [("D2", 0.50), ("D5", 0.33), ("D9", 0.20)]  # 让概念目标获得独立图证据而非重复通用干扰项
queries[2]["channels"]["dense"] = [("D3", 0.88), ("D7", 0.86), ("D1", 0.70)]  # 让关系目标在语义通道也保持最高相关性
queries[2]["channels"]["graph"] = [("D3", 1.00), ("D1", 0.40), ("D8", 0.30)]  # 避免无关退款帮助在三个通道重复累积分数
queries[3]["channels"]["graph"] = [("D4", 0.50), ("D8", 0.45), ("D9", 0.20)]  # 让错误码目标在图诊断关系中得到一致证据
queries[4]["channels"]["graph"] = [("D5", 0.50), ("D9", 0.30), ("D8", 0.20)]  # 为向量扩展目标加入索引依赖关系证据
queries[5]["channels"]["dense"] = [("D6", 0.87), ("D9", 0.85), ("D8", 0.60)]  # 让密钥责任文档在语义通道优先于通用质量文档
queries[5]["channels"]["graph"] = [("D6", 1.00), ("D8", 0.40), ("D5", 0.30)]  # 避免通用文档跨通道重复并突出通知关系
preview = [{"查询": query["id"], "问题": query["text"], "意图": query["intent"], "目标文档": documents[query["expected"]]} for query in queries]  # 汇总预算分配所需的真实语义字段
print("多通道检索输入预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示精确、概念与关系查询的差异

多通道检索输入预览：
[{'查询': 'F01',
  '问题': '订单 EX2048 退款到哪了',
  '意图': 'exact',
  '目标文档': '订单 EX2048 的退款进度'},
 {'查询': 'F02', '问题': '怎样减少 RAG 编造', '意图': 'concept', '目标文档': '用来源引用降低 RAG 幻觉'},
 {'查询': 'F03', '问题': '退款最终需要谁审批', '意图': 'relation', '目标文档': '退款审批人与财务关系'},
 {'查询': 'F04',
  '问题': '错误码 E_CONN_RESET',
  '意图': 'exact',
  '目标文档': 'E_CONN_RESET 连接重置排障'},
 {'查询': 'F05', '问题': '向量检索太慢怎样扩展', '意图': 'concept', '目标文档': '向量索引分片与量化优化'},
 {'查询': 'F06', '问题': '密钥泄露后应该通知谁', '意图': 'relation', '目标文档': '密钥泄露后的通知责任人'}]


## 2. Baseline（基线）：每通道固定 top-2 后直接比较 raw score

这个基线表面上使用了三个通道，却把 BM25 十几分与 cosine 0.9 直接放在一起，最终几乎总选 lexical 第一名。精确查询可能碰巧正确，概念与关系查询会被分数量纲支配。

In [2]:
def raw_score_baseline(query):  # 实现固定预算且直接比较原始分数的错误融合
    candidates = []  # 收集三个通道各自前两名
    for channel in ("lexical", "dense", "graph"):  # 使用不考虑意图的平均通道预算
        for doc_id, raw_score in query["channels"][channel][:2]:  # 从每个通道固定取两个候选
            candidates.append({"doc_id": doc_id, "channel": channel, "raw_score": raw_score})  # 保留不可直接比较的原始分数
    winner = max(candidates, key=lambda item: item["raw_score"])  # 错误地跨量纲选择数值最大的候选
    return winner, candidates  # 返回最终文档与完整候选账本
baseline_rows = []  # 收集六个查询的 raw-score 基线结果
for query in queries:  # 遍历同一批真实查询
    winner, candidates = raw_score_baseline(query)  # 执行固定预算与原始分数比较
    baseline_rows.append({"查询": query["id"], "意图": query["intent"], "基线命中": winner["doc_id"], "标题": documents[winner["doc_id"]], "来源通道": winner["channel"], "raw_score": winner["raw_score"], "目标": query["expected"], "正确": winner["doc_id"] == query["expected"]})  # 保存逐查询基线结果
print("固定 top-2 与 raw-score 基线：")  # 标注当前输出属于错误融合方案
pprint(baseline_rows, sort_dicts=False)  # 展示不同量纲如何让 lexical 支配结果

固定 top-2 与 raw-score 基线：
[{'查询': 'F01',
  '意图': 'exact',
  '基线命中': 'D1',
  '标题': '订单 EX2048 的退款进度',
  '来源通道': 'lexical',
  'raw_score': 12.0,
  '目标': 'D1',
  '正确': True},
 {'查询': 'F02',
  '意图': 'concept',
  '基线命中': 'D9',
  '标题': '大模型回答质量概览',
  '来源通道': 'lexical',
  'raw_score': 10.0,
  '目标': 'D2',
  '正确': False},
 {'查询': 'F03',
  '意图': 'relation',
  '基线命中': 'D7',
  '标题': '通用退款帮助',
  '来源通道': 'lexical',
  'raw_score': 11.0,
  '目标': 'D3',
  '正确': False},
 {'查询': 'F04',
  '意图': 'exact',
  '基线命中': 'D4',
  '标题': 'E_CONN_RESET 连接重置排障',
  '来源通道': 'lexical',
  'raw_score': 15.0,
  '目标': 'D4',
  '正确': True},
 {'查询': 'F05',
  '意图': 'concept',
  '基线命中': 'D8',
  '标题': '数据库常见问题',
  '来源通道': 'lexical',
  'raw_score': 9.0,
  '目标': 'D5',
  '正确': False},
 {'查询': 'F06',
  '意图': 'relation',
  '基线命中': 'D9',
  '标题': '大模型回答质量概览',
  '来源通道': 'lexical',
  'raw_score': 10.0,
  '目标': 'D6',
  '正确': False}]


## 3. 核心机制：按意图分配六个候选预算

精确查询偏向 lexical，概念查询偏向 dense，关系查询偏向 graph；总预算都严格等于 6。每种意图还配置融合权重，但权重作用于 rank reciprocal，而不是原始分数。

In [3]:
budget_by_intent = {"exact": {"lexical": 3, "dense": 2, "graph": 1}, "concept": {"lexical": 1, "dense": 3, "graph": 2}, "relation": {"lexical": 1, "dense": 2, "graph": 3}}  # 定义三类意图的候选预算合同
weight_by_intent = {"exact": {"lexical": 2.0, "dense": 1.0, "graph": 0.5}, "concept": {"lexical": 0.7, "dense": 1.5, "graph": 1.0}, "relation": {"lexical": 0.6, "dense": 1.0, "graph": 1.5}}  # 定义按查询类型校准的通道权重
def select_by_budget(query):  # 根据查询意图从每个通道取得指定数量候选
    budget = budget_by_intent[query["intent"]]  # 读取当前意图的六候选分配
    selected = {channel: query["channels"][channel][:count] for channel, count in budget.items()}  # 截取各通道内部有序候选
    return budget, selected  # 返回可审计预算与候选集合
sample_budget, sample_selected = select_by_budget(queries[2])  # 为退款审批关系查询分配候选预算
print({"关系查询": queries[2]["text"], "预算": sample_budget, "选中候选": {channel: [(doc_id, score) for doc_id, score in candidates] for channel, candidates in sample_selected.items()}})  # 展示图通道获得更多预算的关键中间量

{'关系查询': '退款最终需要谁审批', '预算': {'lexical': 1, 'dense': 2, 'graph': 3}, '选中候选': {'lexical': [('D7', 11.0)], 'dense': [('D3', 0.88), ('D7', 0.86)], 'graph': [('D3', 1.0), ('D1', 0.4), ('D8', 0.3)]}}


## 4. 手写 weighted RRF：按通道 rank 融合并以 doc_id 去重

RRF 得分为各通道 `weight / (k + rank)` 之和。它只依赖通道内部顺序，避免 raw score 量纲冲突；同一 doc 在多个通道出现时累加证据，但最终只占一个文档位置。这里使用较小 k=10 让教学分数差异可见。

In [4]:
def weighted_rrf(query, rrf_k=10):  # 对按预算选出的多通道候选执行加权 RRF
    budget, selected = select_by_budget(query)  # 取得当前查询的意图预算与通道候选
    weights = weight_by_intent[query["intent"]]  # 读取该意图对应的通道可信权重
    fused = {}  # 用 doc_id 聚合跨通道排名证据并天然去重
    ledger = []  # 保存每个通道候选对最终分数的贡献
    for channel, candidates in selected.items():  # 遍历 lexical、dense 与 graph 候选
        for rank, (doc_id, raw_score) in enumerate(candidates, start=1):  # 按通道内部名次处理候选
            contribution = weights[channel] / (rrf_k + rank)  # 将不可比分数转换为可加的倒数排名贡献
            fused[doc_id] = fused.get(doc_id, 0.0) + contribution  # 按稳定文档 ID 合并多个通道证据
            ledger.append({"channel": channel, "rank": rank, "doc": doc_id, "raw": raw_score, "rrf贡献": contribution})  # 保存解释最终排名所需的中间量
    ranked = sorted(fused.items(), key=lambda pair: (-pair[1], pair[0]))  # 按融合分数降序与 doc_id 稳定排序
    return ranked, ledger, budget  # 返回去重全局排名、贡献账本和预算
sample_ranked, sample_ledger, sample_budget = weighted_rrf(queries[2])  # 对关系查询执行完整 RRF 融合
print("退款审批查询的 RRF 贡献账本：")  # 输出核心融合中间量标题
pprint([{**row, "rrf贡献": round(row["rrf贡献"], 5), "标题": documents[row["doc"]]} for row in sample_ledger], sort_dicts=False)  # 展示每个通道 rank 如何贡献最终分数
print("融合排名：", [(doc_id, round(score, 5), documents[doc_id]) for doc_id, score in sample_ranked])  # 展示去重后的全局文档顺序

退款审批查询的 RRF 贡献账本：
[{'channel': 'lexical',
  'rank': 1,
  'doc': 'D7',
  'raw': 11.0,
  'rrf贡献': 0.05455,
  '标题': '通用退款帮助'},
 {'channel': 'dense',
  'rank': 1,
  'doc': 'D3',
  'raw': 0.88,
  'rrf贡献': 0.09091,
  '标题': '退款审批人与财务关系'},
 {'channel': 'dense',
  'rank': 2,
  'doc': 'D7',
  'raw': 0.86,
  'rrf贡献': 0.08333,
  '标题': '通用退款帮助'},
 {'channel': 'graph',
  'rank': 1,
  'doc': 'D3',
  'raw': 1.0,
  'rrf贡献': 0.13636,
  '标题': '退款审批人与财务关系'},
 {'channel': 'graph',
  'rank': 2,
  'doc': 'D1',
  'raw': 0.4,
  'rrf贡献': 0.125,
  '标题': '订单 EX2048 的退款进度'},
 {'channel': 'graph',
  'rank': 3,
  'doc': 'D8',
  'raw': 0.3,
  'rrf贡献': 0.11538,
  '标题': '数据库常见问题'}]
融合排名： [('D3', 0.22727, '退款审批人与财务关系'), ('D7', 0.13788, '通用退款帮助'), ('D1', 0.125, '订单 EX2048 的退款进度'), ('D8', 0.11538, '数据库常见问题')]


## 5. 结果解读：逐查询预算、基线与融合命中

六条查询都保留自己的预算分配，RRF 对精确 ID、概念语义和关系路径采用不同证据权重。结果改进来自通道 rank 和多证据一致性，不是重新缩放某个 raw score。

In [5]:
result_rows = []  # 收集六个查询的意图预算与融合结果
for index, query in enumerate(queries):  # 遍历同一批真实查询
    ranked, ledger, budget = weighted_rrf(query)  # 执行按意图预算和加权 RRF
    winner_id, winner_score = ranked[0]  # 取得去重后的最高融合文档
    result_rows.append({"查询": query["id"], "问题": query["text"], "意图": query["intent"], "预算": budget, "raw基线": baseline_rows[index]["基线命中"], "RRF命中": winner_id, "标题": documents[winner_id], "目标": query["expected"], "正确": winner_id == query["expected"], "融合分": round(winner_score, 5)})  # 保存逐查询完整对照
baseline_hits = sum(row["正确"] for row in baseline_rows)  # 统计直接比较原始分数的命中数量
fusion_hits = sum(row["正确"] for row in result_rows)  # 统计意图预算与 RRF 的命中数量
print("多通道检索逐查询结果：")  # 输出结果解读标题
pprint(result_rows, sort_dicts=False)  # 展示每个查询的预算、基线和融合文档
print({"raw-score基线命中": f"{baseline_hits}/{len(queries)}", "budget+RRF命中": f"{fusion_hits}/{len(queries)}"})  # 汇总同数据上的检索质量差异

多通道检索逐查询结果：
[{'查询': 'F01',
  '问题': '订单 EX2048 退款到哪了',
  '意图': 'exact',
  '预算': {'lexical': 3, 'dense': 2, 'graph': 1},
  'raw基线': 'D1',
  'RRF命中': 'D1',
  '标题': '订单 EX2048 的退款进度',
  '目标': 'D1',
  '正确': True,
  '融合分': 0.31061},
 {'查询': 'F02',
  '问题': '怎样减少 RAG 编造',
  '意图': 'concept',
  '预算': {'lexical': 1, 'dense': 3, 'graph': 2},
  'raw基线': 'D9',
  'RRF命中': 'D2',
  '标题': '用来源引用降低 RAG 幻觉',
  '目标': 'D2',
  '正确': True,
  '融合分': 0.22727},
 {'查询': 'F03',
  '问题': '退款最终需要谁审批',
  '意图': 'relation',
  '预算': {'lexical': 1, 'dense': 2, 'graph': 3},
  'raw基线': 'D7',
  'RRF命中': 'D3',
  '标题': '退款审批人与财务关系',
  '目标': 'D3',
  '正确': True,
  '融合分': 0.22727},
 {'查询': 'F04',
  '问题': '错误码 E_CONN_RESET',
  '意图': 'exact',
  '预算': {'lexical': 3, 'dense': 2, 'graph': 1},
  'raw基线': 'D4',
  'RRF命中': 'D4',
  '标题': 'E_CONN_RESET 连接重置排障',
  '目标': 'D4',
  '正确': True,
  '融合分': 0.31818},
 {'查询': 'F05',
  '问题': '向量检索太慢怎样扩展',
  '意图': 'concept',
  '预算': {'lexical': 1, 'dense': 3, 'graph': 2},
  'raw基线': 'D8',
  'RRF命中': 'D

## 6. 失败案例与修正：graph 为空却仍占三份预算

关系查询若没有识别到实体，graph 通道可能返回空列表。静态预算会白白损失三次候选机会；修正是在不突破总预算 6 的前提下，把未使用份额轮流回填给仍有剩余结果的 dense 和 lexical。

In [6]:
empty_graph_query = {"id": "F07", "text": "新术语之间有什么关系", "intent": "relation", "channels": {"lexical": [("D9", 8.0), ("D5", 6.0), ("D8", 4.0)], "dense": [("D5", 0.90), ("D9", 0.80), ("D8", 0.70)], "graph": []}}  # 构造实体链接失败导致 graph 为空的关系查询
static_budget = budget_by_intent["relation"]  # 读取仍然为 graph 保留三份的静态预算
static_selected = {channel: empty_graph_query["channels"][channel][:count] for channel, count in static_budget.items()}  # 按静态预算截取实际可得候选
static_used = sum(len(candidates) for candidates in static_selected.values())  # 统计空 graph 后真正使用的候选预算
def select_with_backfill(query, total_budget=6):  # 在通道为空或不足时手写预算回填
    budget, selected = select_by_budget(query)  # 先按意图执行正常预算分配
    positions = {channel: len(candidates) for channel, candidates in selected.items()}  # 记录每个通道已经消费到的排名位置
    used = sum(len(candidates) for candidates in selected.values())  # 统计第一轮实际取得的候选数
    while used < total_budget:  # 持续回填直到预算用满或所有通道耗尽
        progressed = False  # 记录本轮是否还有通道可提供候选
        for channel in ("dense", "lexical", "graph"):  # 按关系失败后的回填优先级轮询可用通道
            position = positions[channel]  # 读取当前通道下一候选位置
            if used < total_budget and position < len(query["channels"][channel]):  # 检查通道是否仍有未取候选且总预算未满
                selected[channel].append(query["channels"][channel][position])  # 从该通道补入下一个内部排名候选
                positions[channel] += 1  # 推进当前通道消费位置
                used += 1  # 累加实际使用的总候选预算
                progressed = True  # 标记本轮成功取得至少一个候选
        if not progressed:  # 所有通道都已耗尽时不能继续回填
            break  # 提前结束并把实际候选数交给下游
    return selected, used  # 返回回填后的候选集合与实际预算使用量
backfilled_selected, backfilled_used = select_with_backfill(empty_graph_query)  # 对 graph 空结果执行动态预算回填
print({"失败_静态预算": static_budget, "实际仅使用": static_used, "修正后使用": backfilled_used, "回填候选": {channel: [doc_id for doc_id, score in candidates] for channel, candidates in backfilled_selected.items()}})  # 展示空通道预算浪费和回填结果

{'失败_静态预算': {'lexical': 1, 'dense': 2, 'graph': 3}, '实际仅使用': 3, '修正后使用': 6, '回填候选': {'lexical': ['D9', 'D5', 'D8'], 'dense': ['D5', 'D9', 'D8'], 'graph': []}}


## 7. 生产差距与最小回归检查

生产多通道检索应异步并发，并给每个通道设置延迟 deadline、取消和降级；预算不仅是候选数，还包括毫秒、RPC 和最终上下文 token。意图分类与通道权重需在独立查询集校准，ACL 和时间过滤必须在融合前生效。下面的断言只验证真实六查询、预算总量、raw-score 失败、RRF 命中和空通道回填。

In [7]:
assert len(queries) >= 6  # 确认真实多通道查询数量满足逐样本教学要求
assert all(sum(row["预算"].values()) == 6 for row in result_rows)  # 确认每条主实验查询使用相同总候选预算
assert baseline_hits < fusion_hits  # 确认直接比较不可比 raw score 的效果低于排名融合
assert fusion_hits == len(queries)  # 确认六种精确、概念和关系查询都找回目标文档
assert len({doc_id for doc_id, score in sample_ranked}) == len(sample_ranked)  # 确认同一文档的跨通道证据被融合而非重复占位
assert static_used < 6 and backfilled_used == 6  # 确认空 graph 真实浪费静态预算且动态回填用满预算
print("回归检查通过：意图预算、跨量纲 RRF、doc 去重与空通道回填均已验证。")  # 输出最终验收结论

回归检查通过：意图预算、跨量纲 RRF、doc 去重与空通道回填均已验证。
